# Text conditioning by cross-attention on DeepFashion images, conditioned on captions

Comparison method 3 of 3 on Modality 1: the condition enters the network as text, through
attention, with **no guidance at sampling time**. Loads the class-conditional base shared by every
DeepFashion notebook (or trains it), freezes it, adds one cross-attention block after the
bottleneck self-attention, trains only that block on the **captions**, samples.

**Paper:** Chen, Yu, Ge, Yao, Xie, Wu, Wang, Kwok, Luo, Lu, Li.
*PixArt-α: Fast Training of Diffusion Transformer for Photorealistic Text-to-Image
Synthesis.* ICLR 2024. [arXiv:2310.00426](https://arxiv.org/abs/2310.00426)

**Official code exists** and was read:
[PixArt-alpha/PixArt-alpha](https://github.com/PixArt-alpha/PixArt-alpha),
`MultiHeadCrossAttention` in `PixArt_blocks.py` and its zero-init in `PixArt.py`.
`src/guidance/baselines/cross_attention.py` re-expresses that block in plain PyTorch for
a conv U-Net; its header lists what was kept and what changed.

**The caption is the only route to the garment.** By default the backbone runs with the **null
label** and the cross-attention block reads CLIP token embeddings of the image's caption, which
names the category and attributes (sleeve, neckline, pattern, fabric) the 50-way label does not
carry. Same protocol as the TC-LoRA notebook, so the two arms compare directly; set
`label_path = True` to give the backbone the true class as well.

Run top to bottom. On Colab set **Runtime > Change runtime type > GPU** first. Run
`cfg_fashion.ipynb` first, or place its `checkpoints/diffusion_fashion/best.pt` next to this
notebook, and base training is skipped.

## Protocol shared by every DeepFashion notebook

| | setting |
|---|---|
| base | the U-Net of the Modality-1 base notebook, 64 base channels, verbatim; its weights from `cfg_fashion.ipynb` (`checkpoints/diffusion_fashion/best.pt` or `FASHION_BASE_CKPT`), or trained here with the same recipe if absent |
| base recipe | AdamW 2e-4, constant learning rate, mixed precision, gradient clip 1, 10% label dropout, batch 64, 50 epochs, best validation checkpoint |
| schedule | their cosine schedule, K = 1000, betas clamped to [1e-4, 0.9999], epsilon prediction |
| sampler | deterministic DDIM, 50 steps, clean estimate clamped to [-1, 1], η = 0 |
| condition text | the image's caption from `captions.csv`, frozen CLIP ViT-B/32 |
| evaluation | 2048 class-balanced validation images; one generated image per validation caption or class; scorer = fine-tuned ResNet-18; FID, top-1 / top-5, precision / recall |
| seeds | `seed` for initialisation, training and sampling noise; `split_seed` for the validation subset |

## How this notebook implements the PixArt-α block

**Notation.** $x_k$ is the noisy image at diffusion step $k$ ($K = 1000$; the network receives
$t = k/K$), $\varepsilon_\theta$ the frozen noise-prediction U-Net, $c$ the class, $\varnothing$
the null label, $h(c) \in \mathbb R^{L \times 512}$ the frozen CLIP *token* embeddings of
the image's **caption** with padding mask $M$. $\omega$ are the parameters trained here.

**The paper's idea.** PixArt-α feeds text into a diffusion transformer through a cross-attention
block in every transformer block: the image tokens form the queries, the text tokens the keys
and values, and the result is added back as a plain residual. The block's output projection is
zero-initialised, so a freshly attached block is the identity and training starts from the
unconditioned model:

$$Q = z W_q,\qquad [K\;\;V] = h(c)\, W_{kv},\qquad
\mathrm{Attn}(z, h(c)) = \operatorname{softmax}\!\Big(\frac{Q K^\top}{\sqrt d} + \log M\Big) V$$

$$z' = z + \mathrm{Attn}(z, h(c))\, W_o,\qquad W_o = 0 \text{ at initialisation}$$

**How it is used here.** One such block is attached after the frozen U-Net's bottleneck self-attention,
where the feature map $z \in \mathbb R^{128 \times 16 \times 16}$ is flattened to 256 tokens. Four heads.
Only $W_q, W_{kv}, W_o$ train (the count is printed in *Attach cross-attention*), with the ordinary noise-prediction loss and the null
label, so the text is the model's only route to the class:

$$\mathcal L(\omega) = \mathbb E_{k,\,x_0,\,\varepsilon}\Big[\big\|\varepsilon -
\varepsilon_{\theta,\omega}\big(x_k, k, \varnothing;\ h(c)\big)\big\|_2^2\Big]$$

There is no guidance at sampling: plain DDIM with the conditioned network, $N$ evaluations.
This arm therefore measures what the conditioning mechanism does on its own.

| In PixArt-α (paper and official code) | In this notebook |
|---|---|
| `MultiHeadCrossAttention` in `PixArt_blocks.py`: `q_linear` on image tokens, `kv_linear` on text tokens, one output `proj`, padding mask | `TextCrossAttention` in `cross_attention.py` |
| `proj` zero-initialised in `PixArt.initialize_weights`, so the block starts as the identity | `TextCrossAttention.proj`; the `allclose` assertion in *Attach cross-attention* |
| `x = x + cross_attn(x, y, mask)` in `PixArtBlock.forward`: a plain residual, no norm, no gate | `_Conditioned`, wrapped around the U-Net stage named by `cfg.attach`, inside `TextConditioned` |
| T5-XXL caption tokens through a `y_embedder` MLP | CLIP token embeddings from `build_token_embeddings`, straight into `kv_linear`; no MLP |
| cross-attention in every DiT block | one block, after the bottleneck self-attention (`attach = "attention"`) |
| not in PixArt-α: null label, wrong-text pass, matched validation | `COND`, `xa_val_wrong_text`, `validate`: our evaluation protocol, identical in the TC-LoRA notebook |

**What the paper did that this notebook does not.** Their text encoder is T5-XXL (4096-dim);
ours is CLIP, the same frozen encoder every arm here uses, so the comparison isolates the
mechanism and not the encoder. They place the block in every transformer block and train the
whole 600M-parameter model in stages on 25M images; we attach one block to the frozen
DeepFashion U-Net and train only that block on 209k captioned images. Their condition is a
caption per image, as here. They report results with classifier-free guidance
at scale 4.5; we sample with none. Six deviations are listed in the header of
`src/guidance/baselines/cross_attention.py`.

**Why token embeddings and not a pooled vector.** With a single key the softmax is identically 1
and the block collapses to adding a constant. It would still condition, but it could not select
between words, which is the mechanism under test.

In [ ]:
# Colab bootstrap. Run once.
import importlib.util, subprocess, sys, os

for pkg in ("transformers", "datasets"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

if importlib.util.find_spec("guidance") is None:
    # Inside a checkout (Jupyter on a server, cwd somewhere under the repo): walk up to its src/.
    from pathlib import Path as _P
    _src = next((p / "src" for p in [_P.cwd(), *_P.cwd().parents] if (p / "src" / "guidance").is_dir()), None)
    if _src is None:                                   # Colab: clone the repo next to the notebook
        if not os.path.exists("discrete-generative-models-proj-1"):
            subprocess.run(["git", "clone", "-q",
                            "https://github.com/imperorrp/discrete-generative-models-proj-1.git"],
                           check=True)
        _src = "discrete-generative-models-proj-1/src"
    sys.path.insert(0, str(_src))
    print("guidance package from:", _src)

import torch
from guidance.baselines.cross_attention import TextConditioned

def _git_sha():
    for d in ("discrete-generative-models-proj-1", "."):
        try:
            return subprocess.check_output(["git", "-C", d, "rev-parse", "--short", "HEAD"],
                                           text=True, stderr=subprocess.DEVNULL).strip()
        except Exception:
            pass
    return "unknown"

GIT_SHA = _git_sha()
print("code version:", GIT_SHA)
print("device:", "cuda" if torch.cuda.is_available()
      else "CPU  <-- Runtime > Change runtime type > GPU")

## Data source

DeepFashion, Category and Attribute Prediction benchmark (Liu et al., CVPR 2016), with one
caption per image in `captions.csv`. Unzip `DeepFashion.zip` next to the notebook, or under
`data/` in the repo, or mount Drive; the loader searches those places. Set `FASHION_ROOT` if it
is somewhere else. The folder must contain `img/`, `Anno_coarse/`, `Eval/list_eval_partition.txt`
and `captions.csv`.

## Config

In [ ]:
import math, time, json
from dataclasses import dataclass, asdict, fields
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Cfg:
    seed: int = 42                 # training randomness, as cfg_fashion
    split_seed: int = 42           # validation subset; fixed so multi-seed runs share one subset
    batch_size: int = 64
    base_epochs: int = 50
    xattn_epochs: int = 10
    lr_base: float = 2e-4
    lr_schedule: str = "constant"  # the DeepFashion base recipe
    amp: bool = True               # fp16 autocast on CUDA, as the DeepFashion base recipe
    lr_xattn: float = 1e-4
    K: int = 1000                  # diffusion training steps
    sample_steps: int = 50         # sampling steps
    eta: float = 0.0               # 0 deterministic DDIM; 1 ancestral DDPM
    heads: int = 4
    attach: str = "attention"      # U-Net stage whose output the text conditions: the bottleneck self-attention
    label_path: bool = False       # False: backbone sees the null label; text is the only condition
    reuse_base: bool = True        # load base.pt if present instead of retraining
    limit: int = 0                 # >0 truncates the training set, for smoke tests
    val_limit: int = 2048          # class-balanced validation subset, as the base notebook
    base_channels: int = 64        # width of the DeepFashion U-Net, as the base notebook
    n_eval: int = 2048             # generated images per arm for FID, accuracy, precision / recall
    clf_epochs: int = 10           # fine-tuning epochs of the scorer, the base notebooks' recipe

cfg = Cfg()

# Any field can be overridden from the environment, e.g. CFG_BASE_EPOCHS=1 CFG_LIMIT=5000, so
# headless runs (nbconvert, a server, a different GPU) never need a cell edited.
for _f in fields(Cfg):
    _v = os.environ.get(f"CFG_{_f.name.upper()}")
    if _v is not None:
        _cur = getattr(cfg, _f.name)
        setattr(cfg, _f.name, tuple(float(t) for t in _v.split(",")) if isinstance(_cur, tuple)
                else type(_cur)(_v) if not isinstance(_cur, bool) else _v.lower() in ("1", "true", "yes"))
        print(f"cfg.{_f.name} <- {getattr(cfg, _f.name)!r} (from CFG_{_f.name.upper()})")
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_DIR = os.environ.get("FASHION_CKPT_DIR", "checkpoints")   # shared base and classifier checkpoints

def clf_search(arch):
    '''Where a fine-tuned classifier may already be: this folder's checkpoints/, the repo's (two
    levels up from notebooks/fashion/), data/, or FASHION_CLF_DIR. The flow-matching base notebook
    saves its scorer as fashion_resnet18_classifier.pt; drop it in and no fine-tuning runs.'''
    dirs = [os.environ.get("FASHION_CLF_DIR"), CKPT_DIR, "../checkpoints", "../../checkpoints", "data", "../../data"]
    return [f"{d}/fashion_{arch}_classifier.pt" for d in dirs if d]
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
print(DEV, asdict(cfg))

## Data

The loader is the one from the Modality-1 base notebooks, imported from
`src/guidance/data/fashion.py`: crop to the garment box, pad to a white square, 64×64, random
horizontal flips for training, pixels in `[-1, 1]`, the official train / validation partition,
and a class-balanced validation subset of 2048 images. 46 of the 50 coarse categories have
training images.

Each loader yields `(x, idx)`: the image and its position in its split. Two lookups turn the
position into a condition: `LAB_TR[idx]` / `LAB_VA[idx]` is the **class label** the backbone was
trained on (50 categories, null token 50), and `CAP_TR[idx]` / `CAP_VA[idx]` is the image's
**caption**, e.g. "This is a chic sleeveless pleated blouse with a v-neckline." The caption is
the text the text-conditioned arms read: it names the category *and* attributes (sleeve, neckline,
pattern, fabric) that the 50-way label does not carry. That is the semantic information a
language-model embedding can bring that a label cannot.

In [ ]:
from guidance.data.fashion import make_loaders

train_loader, val_loader, meta = make_loaders(
    batch_size=cfg.batch_size, seed=cfg.split_seed, limit=cfg.limit, val_limit=cfg.val_limit)

LAB_TR, LAB_VA = meta["labels_train"].to(DEV), meta["labels_val"].to(DEV)      # position -> class label
CAP_TR, CAP_VA = meta["captions_train"], meta["captions_val"]                  # position -> caption
ACTIVE = torch.tensor(meta["active_class_ids"], device=DEV)                    # classes with training images
N_VAL = meta["n_val"]

xb, cb = next(iter(train_loader))
print("x", tuple(xb.shape), f"[{float(xb.min()):.2f}, {float(xb.max()):.2f}]", "| idx", tuple(cb.shape))
print(f"train {meta['n_train']:,} | val {meta['n_val']:,} | classes {meta['n_classes']} "
      f"({len(meta['active_class_ids'])} with training images) | root {meta['root']}")
i0 = int(cb[0])
print(f"first item: class '{meta['class_names'][int(LAB_TR[i0])]}' | caption: {CAP_TR[i0]}")

## Frozen text encoder: token embeddings of each caption

Cross-attention needs the **per-token** embeddings and the padding mask: with a single key token
the softmax is identically 1 and the block degenerates into adding a constant. It still
conditions, but it cannot select between words, which is the point of the mechanism. With a
caption of ten to twenty words there is something to select between: "sleeveless", "pleated",
"v-neckline".

The 2048 validation captions are encoded once. Training captions are encoded per batch by the
frozen CLIP text model (209k captions of token embeddings would not fit in memory). Falls back
to random tokens if CLIP will not load; the manifest records that, and a fallback run is not a
result.

In [ ]:
def make_text_encoder(device):
    '''Returns (text_tokens(captions) -> (tokens (B, L, 512), mask (B, L) bool), is_fallback).'''
    try:
        from transformers import CLIPTokenizer, CLIPTextModel, logging as hf_logging
        hf_logging.set_verbosity_error()          # silence the vision_model.* UNEXPECTED report
        name = "openai/clip-vit-base-patch32"
        tok = CLIPTokenizer.from_pretrained(name)
        enc = CLIPTextModel.from_pretrained(name).to(device).eval()
        for p in enc.parameters():
            p.requires_grad_(False)

        @torch.no_grad()
        def text_tokens(captions):
            b = tok(list(captions), padding=True, truncation=True, return_tensors="pt")
            with torch.autocast(device_type="cuda", enabled=False):
                hidden = enc(**{k: v.to(device) for k, v in b.items()}).last_hidden_state.float()
            return hidden, b["attention_mask"].to(device).bool()
        text_tokens(["a photo of a dress"])
        return text_tokens, False
    except Exception as e:
        print(f"CLIP unavailable ({type(e).__name__}); using random fallback tokens.")
        table = torch.randn(len(CAP_TR) + len(CAP_VA), 8, 512, generator=torch.Generator().manual_seed(0)).to(device)
        lookup = {c: i for i, c in enumerate(CAP_TR + CAP_VA)}

        def text_tokens(captions):
            idx = torch.tensor([lookup[c] for c in captions], device=device)
            return table[idx], torch.ones(len(captions), 8, dtype=torch.bool, device=device)
        return text_tokens, True

text_tokens, IS_FALLBACK = make_text_encoder(DEV)
TEXT_VA_TOK, TEXT_VA_MASK = text_tokens(CAP_VA)                 # every validation caption, once
TEXT_DIM = TEXT_VA_TOK.shape[-1]

def text_tr(y):
    '''Token embeddings of the training captions at positions y, encoded on the fly.'''
    return text_tokens([CAP_TR[i] for i in y.tolist()])

print("validation token embeddings:", tuple(TEXT_VA_TOK.shape), "| real tokens per caption:",
      TEXT_VA_MASK.sum(1).tolist()[:5], "... | fallback:", IS_FALLBACK)

## Backbone

In [ ]:
# ---- The model of notebooks/fashion/CIS6720_Proj1_DiffusionCode_Modality_1.ipynb, a teammate's
# class-conditional base for DeepFashion, kept verbatim. The class label is added to the time
# embedding and enters every residual block; bottleneck self-attention at 16x16.
base_channels = cfg.base_channels
num_classes = meta["n_classes"]


def timestep_embedding(k, width):
    '''Sinusoidal timestep features, before a learned projection.'''
    half = width // 2
    frequencies = torch.exp(-math.log(10000) * torch.arange(half, device=k.device)
                            / max(half - 1, 1))
    angles = k.float()[:, None] * frequencies[None]
    result = torch.cat([angles.sin(), angles.cos()], dim=-1)
    return F.pad(result, (0, width - result.shape[-1]))


class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, embedding_width):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.embedding = nn.Linear(embedding_width, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.skip = (nn.Conv2d(in_channels, out_channels, 1)
                     if in_channels != out_channels else nn.Identity())

    def forward(self, x, embedding):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.embedding(F.silu(embedding))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Attention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.out = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        q, k, v = self.qkv(self.norm(x)).chunk(3, dim=1)
        q = q.flatten(2).transpose(1, 2)
        k = k.flatten(2).transpose(1, 2)
        v = v.flatten(2).transpose(1, 2)
        h = F.scaled_dot_product_attention(q, k, v)
        h = h.transpose(1, 2).reshape(batch, channels, height, width)
        return x + self.out(h)


class DiffusionUNet(nn.Module):
    def __init__(self, classes=num_classes, base=base_channels):
        super().__init__()
        self.base = base
        width = base * 4
        self.time_mlp = nn.Sequential(nn.Linear(base, width), nn.SiLU(), nn.Linear(width, width))
        self.class_embedding = nn.Embedding(classes + 1, width)
        self.input = nn.Conv2d(3, base, 3, padding=1)
        self.down1 = ResBlock(base, base, width)
        self.stride1 = nn.Conv2d(base, base, 3, stride=2, padding=1)
        self.down2 = ResBlock(base, base * 2, width)
        self.stride2 = nn.Conv2d(base * 2, base * 2, 3, stride=2, padding=1)
        self.middle1 = ResBlock(base * 2, base * 2, width)
        self.attention = Attention(base * 2)  # Bottleneck self-attention at 16×16.
        self.middle2 = ResBlock(base * 2, base * 2, width)
        self.up2 = ResBlock(base * 4, base * 2, width)
        self.up1 = ResBlock(base * 3, base, width)
        self.output_norm = nn.GroupNorm(8, base)
        self.output = nn.Conv2d(base, 3, 3, padding=1)
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, xk, k, class_labels):
        embedding = (self.time_mlp(timestep_embedding(k, self.base))
                     + self.class_embedding(class_labels))
        x = self.input(xk)
        skip1 = self.down1(x, embedding)              # [B, base, 64, 64]
        skip2 = self.down2(self.stride1(skip1), embedding)  # [B, 2*base, 32, 32]
        x = self.middle2(self.attention(self.middle1(self.stride2(skip2), embedding)), embedding)
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.up2(torch.cat([x, skip2], dim=1), embedding)
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.up1(torch.cat([x, skip1], dim=1), embedding)
        return self.output(F.silu(self.output_norm(x)))  # Predict RGB noise.


# ---- Our wrapper. The notebooks pass t in [0, 1] and an optional class id; None is the null
# token, their `null_class = num_classes`. t is scaled back to the integer step their time
# embedding was written for.
class FashionUNet(DiffusionUNet):
    def __init__(self):
        super().__init__(classes=meta["n_classes"], base=cfg.base_channels)
        self.null = meta["n_classes"]

    def forward(self, x, t, cond_ids=None):
        if cond_ids is None:
            cond_ids = torch.full((x.shape[0],), self.null, device=x.device, dtype=torch.long)
        return super().forward(x, t * cfg.K, cond_ids)


def make_backbone():
    return FashionUNet()

# Only the trunk is adapted by TC-LoRA: the input convolution, every residual block's two 3x3
# convolutions and 1x1 skip, and the attention's qkv / out projections. The time MLP, the class
# embedding, each block's embedding projection (the conditioning path) and the output layer are
# not; the strided downsampling convolutions cannot be (deviation 1).
SKIP_NAMES = ("time_mlp", "class_embedding", ".embedding", "output")

torch.manual_seed(cfg.seed)      # model init and training randomness follow `seed`
net = make_backbone().to(DEV)
print(f"backbone params: {sum(p.numel() for p in net.parameters()):,}")

MID_CHANNELS = 2 * cfg.base_channels   # width of the stage named by cfg.attach, the bottleneck attention

## Diffusion — cosine schedule, epsilon prediction, DDIM sampler

In [ ]:
def cosine_alpha_bar(K, s=0.008, device="cpu"):
    steps = torch.arange(K + 1, dtype=torch.float32, device=device) / K
    f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0])[1:].clamp(1e-5, 0.9999)

def cosine_alpha_bar_diffusers(K, max_beta=0.999, device="cpu"):
    '''diffusers "squaredcos_cap_v2": the same cosine curve turned into betas clipped at
    max_beta, then a cumulative product. The schedule the cell base (cfg_cells) was trained with.'''
    steps = torch.arange(K + 1, dtype=torch.float64, device=device) / K
    f = torch.cos((steps + 0.008) / 1.008 * math.pi / 2) ** 2
    betas = (1 - f[1:] / f[:-1]).clamp(max=max_beta)
    return torch.cumprod(1 - betas, 0).float()

def cosine_alpha_bar_fashion(K, device="cpu"):
    '''The schedule of the DeepFashion base notebook, verbatim: the cosine curve turned into betas
    clamped to [1e-4, 0.9999], then a cumulative product. Sharing the base checkpoint requires
    sampling with the schedule it was trained under.'''
    step_fraction = torch.linspace(0, K, K + 1, device=device, dtype=torch.float64) / K
    cosine_curve = torch.cos((step_fraction + 0.008) / 1.008 * math.pi / 2).square()
    cosine_curve = cosine_curve / cosine_curve[0]
    betas = (1 - cosine_curve[1:] / cosine_curve[:-1]).clamp(0.0001, 0.9999).float()
    return torch.cumprod(1 - betas, dim=0)

ABAR = cosine_alpha_bar_fashion(cfg.K, device=DEV)
NULL_ID = meta["n_conditions"]

# At the first DDIM step sqrt(abar) is about 0.003, so x0 = (x - sqrt(1-abar) eps)/sqrt(abar)
# divides by a tiny number and any error in eps is amplified ~300x. Without clamping the
# samples blow up. Pixels are normalised to [-1, 1], so the clamp is 1.
CLIP_X0 = 1.0


def add_noise(x0, gen=None):
    '''gen: pass a seeded torch.Generator to get the same (k, eps) every call.'''
    n = x0.shape[0]
    k = torch.randint(0, cfg.K, (n,), device=x0.device, generator=gen)
    eps = torch.randn(x0.shape, device=x0.device, generator=gen)
    ab = ABAR[k].view(n, *([1] * (x0.dim() - 1)))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps, k, eps


def drop_labels(y, p_drop=0.1):
    '''Condition dropout creates the null branch that guidance needs later.'''
    keep = torch.rand(y.shape[0], device=y.device) >= p_drop
    return torch.where(keep, y, torch.full_like(y, NULL_ID))


@torch.no_grad()
def ddim_sample(eps_fn, n, shape, steps, clip_x0=None, n_snapshots=0, x_init=None, eta=None, generator=None):
    '''eps_fn(x, t) -> predicted noise. Deterministic DDIM.

    Convention, shared by every arm in this project: clip the predicted x0, then recompute
    eps from the clipped x0 before stepping (as TFG's official sampler does). Keeping the
    original eps is a different update whenever the clamp engages.

    x_init: pass the same starting noise to two calls for a paired comparison; with eta > 0
    also pass identically seeded generators, or the per-step noise differs.
    eta: 0 deterministic DDIM (images), 1 ancestral DDPM (cells, as the base's own notebook).
    n_snapshots > 0 also returns a list of (timestep, image) pairs for the trajectory
    figure: the starting noise, then the model's clean-image estimate x0_hat at that many
    evenly spaced steps. The last pair is the finished sample.
    '''
    clip_x0 = CLIP_X0 if clip_x0 is None else clip_x0
    idx = torch.linspace(cfg.K - 1, 0, steps).long().to(DEV)
    keep = set(torch.linspace(1, steps - 1, max(n_snapshots, 1)).long().tolist())
    eta = cfg.eta if eta is None else eta
    x = torch.randn(n, *shape, device=DEV, generator=generator) if x_init is None else x_init.clone()
    snaps = [(cfg.K, x.cpu().clone())] if n_snapshots else []
    for i, k in enumerate(idx):
        t = (k.float() / cfg.K).repeat(n)
        eps = eps_fn(x, t)
        ab = ABAR[k]
        ab_prev = ABAR[idx[i + 1]] if i + 1 < len(idx) else torch.ones_like(ab)
        x0 = (x - (1 - ab).sqrt() * eps) / ab.sqrt()
        if clip_x0:
            x0 = x0.clamp(-clip_x0, clip_x0)
        eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()          # recompute from (the clipped) x0, as TFG does
        # eta = 0 is deterministic DDIM; eta = 1 is ancestral DDPM (DDIM paper eq. 16), with fresh
        # noise at every step drawn from `generator` so paired rows stay paired.
        sigma = eta * ((1 - ab_prev) / (1 - ab) * (1 - ab / ab_prev)).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev - sigma ** 2).clamp(min=0).sqrt() * eps
        if eta > 0:
            x = x + sigma * torch.randn(x.shape, device=DEV, generator=generator)
        if n_snapshots and i in keep:
            # x0_hat, not x_t: the noisy state looks like noise until the last few steps,
            # while the clean estimate shows what the model believes at every step.
            snaps.append((int(k), x0.cpu().clone()))
    return (x, snaps) if n_snapshots else x

## Base model: load the shared DeepFashion base, or train it

The base is trained once by `cfg_fashion.ipynb`, the Modality-1 base notebook with our evaluation
appended, and saved as `checkpoints/diffusion_fashion/best.pt`. This cell loads it (or `base.pt`
from an earlier run, after checking split, schedule and backbone), so the three comparison
notebooks and the classifier-free-guidance notebook sit on literally the same weights. If neither
is present it trains the same recipe here. Training uses 10% label dropout; validation does not,
and scores the same fixed noise and timesteps every epoch with the true label.

In [ ]:
VAL_SEED = cfg.seed + 999


def run_epochs(model, params, n_epochs, lr, tag, train_fn, val_fn, ckpt):
    '''Trains with train_fn, validates with val_fn on fixed noise, keeps the best checkpoint.'''
    # cfg.lr_schedule: "cosine" anneals to zero over the run (images); "constant" holds lr, the
    # cell base's recipe. cfg.amp: fp16 autocast with a grad scaler on CUDA, also the cell base's
    # recipe; the loss itself is computed in float32 either way.
    opt = torch.optim.AdamW(params, lr=lr)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(n_epochs, 1)) if cfg.lr_schedule == "cosine"
             else torch.optim.lr_scheduler.LambdaLR(opt, lambda _: 1.0))
    use_amp = bool(cfg.amp) and DEV.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    hist, best, best_ep = [], float("inf"), 0
    for ep in range(1, n_epochs + 1):
        model.train(); tot = num = 0; t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEV), y.to(DEV)
            with torch.autocast(device_type="cuda", enabled=use_amp):
                loss = train_fn(x, y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update()
            tot += loss.item() * x.shape[0]; num += x.shape[0]
        sched.step()

        val = validate(model, val_fn)
        is_best = val < best
        if is_best:
            best, best_ep = val, ep
            torch.save({"state": model.state_dict(), "epoch": ep, "val": val}, ckpt)
        hist.append({"epoch": ep, "train": tot / num, "val": val, "best": is_best,
                     "seconds": time.time() - t0})
        print(f"{tag} epoch {ep:3d}  train {tot/num:.4f}  val {val:.4f}"
              f"{'  *best' if is_best else ''}  ({time.time()-t0:.0f}s)")
    print(f"{tag}: best val {best:.4f} at epoch {best_ep}, saved to {ckpt}")
    return hist


@torch.no_grad()
def validate(model, val_fn):
    '''Same examples, same timesteps, same noise on every call -- for any model.'''
    model.eval()
    gen = torch.Generator(device=DEV).manual_seed(VAL_SEED)
    tot = num = 0
    for x, y in val_loader:
        x, y = x.to(DEV), y.to(DEV)
        tot += val_fn(x, y, gen).item() * x.shape[0]; num += x.shape[0]
    return tot / num


def base_train(x, y):
    # y is the position in the training split; the backbone is trained on the class label with
    # 10% label dropout, the recipe of the DeepFashion base notebook.
    xk, k, eps = add_noise(x)
    return F.mse_loss(net(xk, k.float() / cfg.K, drop_labels(LAB_TR[y])), eps)


def base_val(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(net(xk, k.float() / cfg.K, LAB_VA[y]), eps)     # no dropout


# What a base.pt must agree on before it can be reused. A checkpoint trained under a
# different split would put its own training images into this run's validation set.
CKPT_META = {"split_seed": cfg.split_seed, "n_conditions": meta["n_conditions"],
             "shape": list(meta["shape"]), "K": cfg.K, "schedule": "cosine_fashion",
             "backbone_params": sum(p.numel() for p in net.parameters())}

if cfg.reuse_base and os.path.exists("base.pt"):
    ck = torch.load("base.pt", map_location=DEV)
    saved = ck.get("meta")
    if saved is None:
        print("WARNING: base.pt carries no setup metadata (older run); split and schedule "
              "cannot be verified. Treat any comparison against it with care.")
    else:
        mismatch = {k: (saved.get(k), v) for k, v in CKPT_META.items() if saved.get(k) != v}
        assert not mismatch, f"base.pt was trained under a different setup: {mismatch}"
    net.load_state_dict(ck["state"])
    base_hist = [{"epoch": ck["epoch"], "train": float("nan"), "val": ck["val"], "best": True}]
    print(f"loaded base.pt (epoch {ck['epoch']}, val {ck['val']:.4f}); skipping base training")
elif cfg.reuse_base and (_cfg_ckpt := next(
        (p for p in [os.environ.get("FASHION_BASE_CKPT"), f"{CKPT_DIR}/diffusion_fashion/best.pt",
                     "../checkpoints/diffusion_fashion/best.pt", "../../checkpoints/diffusion_fashion/best.pt"]
         if p and os.path.exists(p)), None)):
    # The base trained by cfg_fashion.ipynb: same model class, so the weights load directly. Every
    # DeepFashion notebook then sits on literally the same base. Its validation loss was computed with
    # that notebook's noise draws, so it is reported as it was recorded, not recomputed.
    ck = torch.load(_cfg_ckpt, map_location=DEV, weights_only=False)
    assert ck["meta"]["n_params"] == sum(p.numel() for p in net.parameters()) and \
        ck["meta"]["num_train_timesteps"] == cfg.K, f"{_cfg_ckpt} does not match this backbone"
    net.load_state_dict(ck["model"])
    base_hist = [{"epoch": ck["epoch"], "train": float("nan"), "val": ck["val_loss"], "best": True}]
    torch.save({"state": net.state_dict(), "epoch": ck["epoch"], "val": ck["val_loss"], "meta": CKPT_META,
                "source": _cfg_ckpt}, "base.pt")
    print(f"loaded the cfg_fashion base from {_cfg_ckpt} (epoch {ck['epoch']}, val {ck['val_loss']:.4f}); "
          f"saved as base.pt; skipping base training")
else:
    base_hist = run_epochs(net, list(net.parameters()), cfg.base_epochs, cfg.lr_base,
                           "base", base_train, base_val, "base.pt")
    ck = torch.load("base.pt", map_location=DEV); ck["meta"] = CKPT_META; torch.save(ck, "base.pt")

## Attach cross-attention and train only that block

The backbone is frozen. One `TextCrossAttention` block is inserted after the stage named
by `cfg.attach`; it flattens the feature map into query tokens, attends to the text tokens,
and adds the result back. Its output projection is zero-initialised (PixArt-α's own
initialisation), so the first assertion — adapted model equals base at step zero — is what
makes this a controlled comparison.

`COND(label)` is the label the backbone sees: the true class if `cfg.label_path`, otherwise the
null token. With the null token, the caption is the model's only knowledge of the garment.

In [ ]:
ckpt = torch.load("base.pt", map_location=DEV)
base_frozen = make_backbone().to(DEV)
base_frozen.load_state_dict(ckpt["state"])
base_frozen.eval()

def COND(y):
    '''The label the frozen backbone sees. Null by default, so the class can only reach the
    model through the text. cfg.label_path = True gives the backbone the true label.'''
    return y if cfg.label_path else torch.full_like(y, NULL_ID)

probe_x, probe_y = xb[:4].to(DEV), cb[:4].to(DEV)
probe_t = torch.rand(4, device=DEV)
with torch.no_grad():
    ref = base_frozen(probe_x, probe_t, COND(LAB_TR[probe_y])).clone()
probe_tok, probe_mask = text_tr(probe_y)

# PixArtBlock.forward: x = x + cross_attn(x, text), here after the U-Net stage named by
# cfg.attach. Queries come from the feature map, keys and values from the text tokens.
xa = TextConditioned(base_frozen, attach=cfg.attach, channels=MID_CHANNELS,
                     text_dim=TEXT_DIM, heads=cfg.heads).to(DEV)

# PixArt.initialize_weights: the output projection starts at zero, so the block is the
# identity at this point and everything it later does is measured from the base model.
with torch.no_grad():
    got = xa(probe_x, probe_t, probe_tok, probe_mask, cond_ids=COND(LAB_TR[probe_y]))
assert torch.allclose(got, ref, atol=1e-5), (got - ref).abs().max().item()
print("no-op at init confirmed")

assert all(".attn." in n for n, p in xa.named_parameters() if p.requires_grad)
# The adapter now lives INSIDE base_frozen, so count frozen params by requires_grad, not
# by module membership -- otherwise the adapter is counted on both sides.
N_FROZEN = sum(p.numel() for p in base_frozen.parameters() if not p.requires_grad)
print(f"trainable: {xa.n_trainable():,} params, all in the cross-attention block")
print(f"frozen:    {N_FROZEN:,} params")
print(f"label path: {'ON (true label)' if cfg.label_path else 'OFF (null label; text only)'}")

In [ ]:
def xa_train(x, y):
    # The ordinary noise-prediction loss. Only the attention block has requires_grad; the
    # U-Net and CLIP stay fixed. No guidance anywhere: the text acts only through attention.
    # y is the position in the split: text_tr(y) / TEXT_VA_*[y] are the caption's tokens, LAB_*[y] the class.
    xk, k, eps = add_noise(x)
    tok, mask = text_tr(y)
    return F.mse_loss(xa(xk, k.float() / cfg.K, tok, mask, cond_ids=COND(LAB_TR[y])), eps)


def xa_val(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(xa(xk, k.float() / cfg.K, TEXT_VA_TOK[y], TEXT_VA_MASK[y], cond_ids=COND(LAB_VA[y])), eps)


xa_hist = run_epochs(xa, [p for p in xa.parameters() if p.requires_grad],
                     cfg.xattn_epochs, cfg.lr_xattn, "xattn", xa_train, xa_val, "xattn.pt")

best = torch.load("xattn.pt", map_location=DEV)["state"]
xa.load_state_dict(best)
xa.eval()
print("restored best cross-attention block")

## The comparable numbers, and the diagnostic that matters

Three validation MSEs on identical examples, timesteps and noise:

- **base, same label path** — what the text-conditioned model starts from. With the label
  path off this is the *unconditional* model.
- **base, true label** — a reference: what knowing the class is worth when it comes in
  through the label embedding.
- **cross-attention** — the arm under test.

And the diagnostic: validation with the **wrong text**. Positions are shifted by a fixed
offset, so every example gets another image's caption, and the shift consumes no
random state, so correct- and wrong-text passes see byte-identical noise and timesteps.
(The validation subset is class-balanced and interleaved, so a shift almost always lands on
a different category as well as a different caption.) Averaged over three offsets.

Read the result as *average loss sensitivity to the text*. A small aggregate difference
is a finding, not proof the text is ignored — it could matter only at some noise levels,
or only for the samples, which the next cells check separately. A null here is
reportable. Base-vs-adapter differences include the adaptation itself; correct-vs-wrong
text *within* the adapted model is what isolates the text.

In [ ]:
def frozen_val_same_path(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(base_frozen(xk, k.float() / cfg.K, COND(LAB_VA[y])), eps)


def frozen_val_label(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(base_frozen(xk, k.float() / cfg.K, LAB_VA[y]), eps)


def xa_val_wrong_text(offset):
    '''Wrong text via a fixed label shift: always a different class, never touches gen.
    Only the text changes. Noise, timesteps and the backbone's label input are identical to
    the correct-text pass, so any loss difference is the text.'''
    def fn(x, y, gen):
        xk, k, eps = add_noise(x, gen)
        y_wrong = (y + offset) % N_VAL                         # another validation image's caption
        return F.mse_loss(xa(xk, k.float() / cfg.K, TEXT_VA_TOK[y_wrong], TEXT_VA_MASK[y_wrong],
                             cond_ids=COND(LAB_VA[y])), eps)
    return fn


WRONG_OFFSETS = (1, 37, 73) if N_VAL > 73 else (1, 2, 3)

VAL_BASE_SAME = validate(base_frozen, frozen_val_same_path)
VAL_BASE_LABEL = validate(base_frozen, frozen_val_label)
VAL_XA = validate(xa, xa_val)
VAL_XA_WRONG_EACH = [validate(xa, xa_val_wrong_text(o)) for o in WRONG_OFFSETS]
VAL_XA_WRONG = sum(VAL_XA_WRONG_EACH) / len(VAL_XA_WRONG_EACH)

print(f"base, same label path as xattn : {VAL_BASE_SAME:.4f}")
print(f"base, true label (reference)   : {VAL_BASE_LABEL:.4f}")
print(f"xattn, correct text            : {VAL_XA:.4f}   ({(VAL_BASE_SAME - VAL_XA) / VAL_BASE_SAME:+.1%} vs base)")
print(f"xattn, wrong text (mean of {len(WRONG_OFFSETS)}) : {VAL_XA_WRONG:.4f}   "
      f"per offset {[round(v, 4) for v in VAL_XA_WRONG_EACH]}")
TEXT_EFFECT = (VAL_XA_WRONG - VAL_XA) / VAL_XA
print()
print(f"text effect (wrong - correct) / correct = {TEXT_EFFECT:+.2%}")
if abs(TEXT_EFFECT) < 0.005:
    print("little average loss sensitivity to the text at this training budget. "
          "Check the per-noise-level breakdown and the sample comparison below before concluding.")

## Text sensitivity by noise level

The aggregate number above averages over the whole trajectory. Text may matter only late
(fine detail) or only early (layout). This bins the correct-vs-wrong loss gap by noise
level on the same fixed noise.

In [ ]:
@torch.no_grad()
def loss_by_noise_level(offset, n_bins=5):
    xa.eval()
    gen = torch.Generator(device=DEV).manual_seed(VAL_SEED)
    edges = torch.linspace(0, cfg.K, n_bins + 1, device=DEV)
    sums = torch.zeros(n_bins, 2, device=DEV); counts = torch.zeros(n_bins, device=DEV)
    for x, y in val_loader:
        x, y = x.to(DEV), y.to(DEV)
        xk, k, eps = add_noise(x, gen)
        t = k.float() / cfg.K
        yw = (y + offset) % N_VAL
        e_ok = ((xa(xk, t, TEXT_VA_TOK[y], TEXT_VA_MASK[y], cond_ids=COND(LAB_VA[y])) - eps) ** 2).flatten(1).mean(1)
        e_wr = ((xa(xk, t, TEXT_VA_TOK[yw], TEXT_VA_MASK[yw], cond_ids=COND(LAB_VA[y])) - eps) ** 2).flatten(1).mean(1)
        b = torch.bucketize(k.float(), edges[1:-1])
        sums.index_add_(0, b, torch.stack([e_ok, e_wr], 1)); counts.index_add_(0, b, torch.ones_like(b, dtype=torch.float))
    return edges.cpu(), (sums / counts[:, None]).cpu()

edges, per_bin = loss_by_noise_level(WRONG_OFFSETS[0])
print(f"{'noise level (k)':>18} {'correct':>9} {'wrong':>9} {'gap':>8}")
TEXT_EFFECT_BY_BIN = []
for i in range(len(per_bin)):
    ok, wr = per_bin[i].tolist()
    TEXT_EFFECT_BY_BIN.append((wr - ok) / ok)
    print(f"{int(edges[i]):>7}-{int(edges[i+1]):<9} {ok:>9.4f} {wr:>9.4f} {(wr-ok)/ok:>+8.2%}")

## Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a, (name, hist) in zip(ax, [("base", base_hist), ("cross-attention", xa_hist)]):
    ep = [h["epoch"] for h in hist]
    if len(hist) > 1:
        a.plot(ep, [h["train"] for h in hist], label="train")
    a.plot(ep, [h["val"] for h in hist], label="val", marker="o" if len(hist) == 1 else None)
    bst = min(hist, key=lambda h: h["val"])
    a.axvline(bst["epoch"], ls="--", c="grey", lw=1)
    a.scatter([bst["epoch"]], [bst["val"]], c="red", zorder=5,
              label=f"selected (epoch {bst['epoch']})")
    a.set_title(f"{name}: best val {bst['val']:.4f}")
    a.set_xlabel("epoch"); a.set_ylabel("noise-prediction MSE"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Samples: same starting noise, three conditions

Eight validation images spread over the subset. Row 1: the base model, same label path as the
arm under test. Row 2: cross-attention with the image's **own caption**. Row 3: cross-attention
with the **next image's caption**. All three start from the same noise, so any difference is the
text. The captions are printed below the grid; the column titles are the class of the caption's
image.

In [ ]:
N = 8
y_eval = torch.linspace(0, N_VAL - 1, N).long().to(DEV)          # positions in the validation subset
y_wrong = (y_eval + 1) % N_VAL
g = torch.Generator(device=DEV).manual_seed(cfg.seed + 1)
X0 = torch.randn(N, *meta["shape"], device=DEV, generator=g)

def paired_gen():
    '''Fresh generator, same seed: identical per-step noise for every row of a paired comparison.'''
    return torch.Generator(device=DEV).manual_seed(cfg.seed + 2)

base_s = ddim_sample(lambda x, t: base_frozen(x, t, COND(LAB_VA[y_eval])), N, meta["shape"],
                     cfg.sample_steps, x_init=X0, generator=paired_gen())
xa_s = ddim_sample(lambda x, t: xa(x, t, TEXT_VA_TOK[y_eval], TEXT_VA_MASK[y_eval], cond_ids=COND(LAB_VA[y_eval])),
                   N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen())
xa_wrong = ddim_sample(lambda x, t: xa(x, t, TEXT_VA_TOK[y_wrong], TEXT_VA_MASK[y_wrong], cond_ids=COND(LAB_VA[y_eval])),
                       N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen())
print("sampled:", tuple(base_s.shape), "| NFE per sample:", cfg.sample_steps, "| paired: same X0 for all rows")

def short(i):
    '''Class of the validation image at position i.'''
    return meta["class_names"][int(LAB_VA[int(i)])]

fig, axes = plt.subplots(3, N, figsize=(2 * N, 6.6))
rows = [("base", base_s, y_eval), ("xattn, own caption", xa_s, y_eval),
        ("xattn, WRONG caption", xa_wrong, y_wrong)]
for r, (name, s, ys) in enumerate(rows):
    for j in range(N):
        axes[r, j].imshow(((s[j].cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[r, j].set_xticks([]); axes[r, j].set_yticks([])
        if r == 1:
            axes[r, j].set_title(short(ys[j]), fontsize=8)
        if r == 2:
            axes[r, j].set_xlabel(short(ys[j]), fontsize=7)
    axes[r, 0].set_ylabel(name, fontsize=10)
plt.tight_layout(); plt.show()
for j in range(N):
    print(f"col {j + 1}: {CAP_VA[int(y_eval[j])]}")

SAMPLE_TEXT_DIFF = ((xa_s - xa_wrong).flatten(1).norm(dim=1) / xa_s.flatten(1).norm(dim=1)).mean().item()
print(f"mean relative difference between correct-text and wrong-text samples: {SAMPLE_TEXT_DIFF:.1%}")

## Denoising trajectory

Four samples along the reverse process. The first column is the starting noise; the rest
show the model's **clean-image estimate** $\hat x_0$ at evenly spaced steps, labelled by
timestep $k$, ending at the finished sample. The noisy state $x_t$ itself looks like noise
until the last few steps, which is why it is not what is shown.

In [ ]:
traj_y = y_eval[:4]
_, snaps = ddim_sample(lambda x, t: xa(x, t, TEXT_VA_TOK[traj_y], TEXT_VA_MASK[traj_y], cond_ids=COND(LAB_VA[traj_y])),
                       4, meta["shape"], cfg.sample_steps, n_snapshots=6)

fig, axes = plt.subplots(4, len(snaps), figsize=(1.6 * len(snaps), 6.6))
for r in range(4):
    for c, (k, s) in enumerate(snaps):
        axes[r, c].imshow(((s[r] + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0:
            axes[r, c].set_title("noise $x_T$" if c == 0 else
                                 ("sample" if c == len(snaps) - 1 else f"$\\hat x_0$, k={k}"), fontsize=8)
    axes[r, 0].set_ylabel(short(traj_y[r]), fontsize=7)
plt.tight_layout(); plt.show()

## Scorer: a classifier the sampler never used

No pretrained DeepFashion classifier exists, so an ImageNet-pretrained ResNet-18 is fine-tuned on
the real training images, once, with the recipe of the flow-matching base notebook (AdamW 1e-3,
cosine annealing, 10 epochs, mixed precision) and cached under `checkpoints/`. It never sees a
generated image during training. Its top-1 / top-5 on real validation images is the ceiling for
the accuracy numbers below; its penultimate features are the space for precision and recall.
Every DeepFashion notebook scores with this same network, so accuracies compare across arms.

In [ ]:
from guidance.eval.image_metrics import fit_classifier, real_accuracy, top_k_hits, precision_recall_features, to_uint8

eval_clf = fit_classifier("resnet18", meta["n_classes"], train_loader, val_loader, LAB_TR, LAB_VA, DEV,
                          f"{CKPT_DIR}/fashion_resnet18_classifier.pt", epochs=cfg.clf_epochs, seed=cfg.seed,
                          search=clf_search("resnet18"))
for p in eval_clf.parameters():
    p.requires_grad_(False)
SCORER_REAL_TOP1, SCORER_REAL_TOP5 = real_accuracy(eval_clf, val_loader, LAB_VA, DEV)
print(f"scorer (fine-tuned ResNet-18) on {N_VAL} real validation images: "
      f"top-1 {SCORER_REAL_TOP1:.3f}, top-5 {SCORER_REAL_TOP5:.3f} (the ceiling; chance {1 / len(ACTIVE):.3f})")

@torch.no_grad()
def clf_features(x):
    return torch.cat([eval_clf.features(x[i:i + 256]) for i in range(0, len(x), 256)])

@torch.no_grad()
def hits(x, y):
    '''top-1 / top-5 hit tensors under the scorer, in batches.'''
    h1, h5 = [], []
    for i in range(0, len(x), 256):
        a, b = top_k_hits(eval_clf(x[i:i + 256]), y[i:i + 256])
        h1.append(a); h5.append(b)
    return torch.cat(h1), torch.cat(h5)

REAL_FEATS = torch.cat([clf_features(x.to(DEV)) for x, _ in val_loader])

## Evaluation on the validation set: accuracy, FID, precision and recall

The protocol of the DeepFashion base notebooks, applied per arm. One image is generated for each
of the `n_eval` validation images, conditioned on **that image's caption** (correct text), on the
**next image's caption** (wrong text), or on nothing (the base with the null label). Then:

- **Condition accuracy**: the scorer's top-1 / top-5 against the class of the caption's image.
  For the wrong-text row the target is the class of the caption actually given. Chance is one
  over the number of active classes; the scorer's accuracy on real images is the ceiling.
- **FID** (torchmetrics, Inception pool features) against the same validation images. With
  2048 on each side FID is biased upward; the count is recorded next to the number.
- **Precision and recall** in the scorer's feature space (k = 3 nearest neighbours): the share
  of generated images inside the real manifold, and the share of real images the generated
  manifold covers.

Same starting noise and same per-step generator for every arm, so the rows are paired. `n_eval`
at 2048 is the full protocol; smaller values are for smoke runs and are not results.

In [ ]:
def _sample(eps_fn, idx, x0, gen):
    return ddim_sample(eps_fn, len(idx), meta["shape"], cfg.sample_steps, x_init=x0, generator=gen)

ARMS = [
    ("base",        lambda idx, x0, gen: _sample(lambda x, t: base_frozen(x, t, COND(LAB_VA[idx])), idx, x0, gen),
                    lambda idx: LAB_VA[idx]),
    ("xattn",       lambda idx, x0, gen: _sample(lambda x, t: xa(x, t, TEXT_VA_TOK[idx], TEXT_VA_MASK[idx], cond_ids=COND(LAB_VA[idx])), idx, x0, gen),
                    lambda idx: LAB_VA[idx]),
    ("xattn_wrong", lambda idx, x0, gen: _sample(lambda x, t: xa(x, t, TEXT_VA_TOK[(idx + 1) % N_VAL], TEXT_VA_MASK[(idx + 1) % N_VAL],
                                                                  cond_ids=COND(LAB_VA[idx])), idx, x0, gen),
                    lambda idx: LAB_VA[(idx + 1) % N_VAL]),        # scored against the caption actually given
]

EVAL_IDX = torch.arange(min(cfg.n_eval, N_VAL), device=DEV)
g = torch.Generator(device=DEV).manual_seed(cfg.seed + 3)
X0_EVAL = torch.randn(len(EVAL_IDX), *meta["shape"], device=DEV, generator=g)

def eval_gen():
    return torch.Generator(device=DEV).manual_seed(cfg.seed + 4)

def generate(sample_fn):
    '''All n_eval images from the same starting noise, in batches, one generator stream.'''
    out, gen = [], eval_gen()
    for i in range(0, len(EVAL_IDX), 128):
        out.append(sample_fn(EVAL_IDX[i:i + 128], X0_EVAL[i:i + 128], gen))
    return torch.cat(out)

def fid_of(x):
    import importlib.util, subprocess, sys
    for pkg, mod in [("torchmetrics", "torchmetrics"), ("torch-fidelity", "torch_fidelity")]:
        if importlib.util.find_spec(mod) is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    from torchmetrics.image.fid import FrechetInceptionDistance
    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(DEV)
    fid.set_dtype(torch.float64)
    for xr, _ in val_loader:
        fid.update(to_uint8(xr.to(DEV)), real=True)
    for i in range(0, len(x), 256):
        fid.update(to_uint8(x[i:i + 256]), real=False)
    return float(fid.compute())

METRICS = {"n_eval": int(len(EVAL_IDX)), "n_real": N_VAL, "scorer": "fine-tuned ResNet-18 (never used by the sampler)",
           "scorer_real_val_top1": SCORER_REAL_TOP1, "scorer_real_val_top5": SCORER_REAL_TOP5,
           "precision_recall_k": 3, "arms": {}}
EVAL_SAMPLES = {}
print(f"{'arm':>14} {'top-1':>7} {'top-5':>7} {'FID':>8} {'prec':>6} {'rec':>6}   {'s/sample':>8}   (n = {len(EVAL_IDX)}; "
      f"real top-1 {SCORER_REAL_TOP1:.3f}; chance {1 / len(ACTIVE):.3f})")
for name, sample_fn, target_fn in ARMS:
    if DEV.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    x = generate(sample_fn)
    if DEV.type == "cuda": torch.cuda.synchronize()
    secs = (time.time() - t0) / len(EVAL_IDX)
    y = target_fn(EVAL_IDX)
    h1, h5 = hits(x, y)
    prec, rec = precision_recall_features(REAL_FEATS, clf_features(x), k=3)
    fid = fid_of(x) if cfg.n_eval >= 64 else float("nan")       # Inception statistics need a real sample size
    row = {"top1": h1.float().mean().item(), "top5": h5.float().mean().item(), "fid": fid,
           "precision": prec, "recall": rec, "seconds_per_sample": secs}
    METRICS["arms"][name] = row
    EVAL_SAMPLES[name] = (x[:8].cpu(), y[:8].cpu())
    print(f"{name:>14} {row['top1']:>7.3f} {row['top5']:>7.3f} {row['fid']:>8.2f} {prec:>6.3f} {rec:>6.3f}   {secs:>8.3f}")
if len(EVAL_IDX) < 2048:
    print("\nNOTE: fewer than 2048 images; FID is biased upward and accuracy is noisy. Not a result.")

## Manifest

In [ ]:
report = {
    "method": "text conditioning by cross-attention (PixArt-alpha block, arXiv:2310.00426)",
    "modality": "images (DeepFashion 64x64, per-image captions)",
    "code_version": GIT_SHA,
    "config": asdict(cfg),
    "label_path": cfg.label_path,
    "val_matched_base_same_path": VAL_BASE_SAME,
    "val_matched_base_true_label": VAL_BASE_LABEL,
    "val_matched_xattn": VAL_XA,
    "val_matched_xattn_wrong_text": VAL_XA_WRONG,
    "val_matched_xattn_wrong_text_per_offset": dict(zip(map(str, WRONG_OFFSETS), VAL_XA_WRONG_EACH)),
    "text_effect_rel": TEXT_EFFECT,
    "text_effect_rel_by_noise_bin": TEXT_EFFECT_BY_BIN,
    "sample_text_diff_rel": SAMPLE_TEXT_DIFF,
    "base_best_val_during_training": min(h["val"] for h in base_hist),
    "xattn_best_val_during_training": min(h["val"] for h in xa_hist),
    "trainable_xattn": xa.n_trainable(),
    "frozen_backbone": N_FROZEN,
    "checkpoint_meta": CKPT_META,
    "attach": cfg.attach,
    "nfe": cfg.sample_steps,
    "clip_fallback": IS_FALLBACK,
    "train_seconds": {"base": sum(h.get("seconds", 0) for h in base_hist),
                      "xattn": sum(h.get("seconds", 0) for h in xa_hist)},
    "hardware": torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu",
    "device": str(DEV),
}
report.update(globals().get("METRICS", {}))
print(json.dumps(report, indent=2))
with open("manifest_xattn_fashion.json", "w") as f:
    json.dump(report, f, indent=2)

## Before reporting any of this

- [ ] `clip_fallback: false` in the manifest
- [ ] `text_effect_rel` reported with its sign and the per-noise-level breakdown; a small
      or null effect is a reportable result, not a reason to withhold the row
- [ ] `n_eval` at 2048 (the base notebooks' protocol); FID at that count is biased upward, say so, and read it with top-1 and precision / recall, never alone
- [ ] The base is the shared `checkpoints/diffusion_fashion/best.pt`; say so, or say it was retrained here
- [ ] Three seeds (`seed`), mean +/- std, with `split_seed` held fixed
- [ ] `label_path` stated next to every number; the two settings are different experiments
- [ ] Described as **a frozen U-Net with a PixArt-α-inspired residual cross-attention
      adapter** — CLIP not T5, one block not every block, attention-only training, no CFG
      — not as PixArt-α reproduced